In [1]:
import warnings
warnings.filterwarnings('ignore')

# 주요 LLM 공급자 및 RAG 답변 평가

# 환경 설정

## 라이브러리 설치

!pip install "langchain>=0.3.0,<0.4.0" "langchain-core>=0.3.0,<0.4.0" "langchain-community>=0.3.0,<0.4.0" "langchain-text-splitters>=0.3.0,<0.4.0" "langchain-experimental>=0.3.0,<0.4.0" "langchain-openai" "langchain-anthropic" "langchain-chroma" "langchain-huggingface" "langchain-ollama" "langchain-google-genai" "langchain-groq" "pydantic>=2.7.0,<3.0.0" "anthropic>=0.30.0" google-genai groq krag kiwipiepy rank_bm25 jq sentence-transformers

`langchain-anthropic`: Anthropic(Claude), 코딩, 논리적 추론, 긴 문맥 처리에 강점이 있는 모델을 사용하기 위해 import 한다.  
`langchain-google-genai`: Google(Gemini), 구글의 최신 모델인 Gemini Pro, Flash 등을 사용하기 위해 import 한다.  
`langchain-groq`: 초고속 추론 엔진인 Groq을 통해 Llama 3, Mixtral 등을 사용하기 위해 import 한다.

## 기본 라이브러리

In [2]:
import os, json, re
from glob import glob
from pprint import pprint
import numpy as np
import pandas as pd

from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import JSONLoader
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
# from langchain_community.embeddings import HuggingFaceEmbeddings # langchain-core 0.3.86일 경우 import 방법
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

## .env 환경 변수

In [3]:
from dotenv import load_dotenv
load_dotenv()

print('LangSmith 추적 여부:', os.getenv('LANGCHAIN_TRACING_V2'))

LangSmith 추적 여부: false


# 실습 환경 준비 및 검색기 초기화

# 데이터를 읽어온다.

## JSONL 파일 읽기

JSONL(JSON Line) 형식의 데이터를 읽어서, LangChain의 표준으로 사용하는 Document 객체 형태로 변환한다.

LangChain의 가장 기본적인 데이터 단위인 본문(page_content)과 부가 정보(metadata)를 묶어서 관리하는 Document를 import 한다.  
`from langchain_core.documents import Document`  
JSON 또는 JSONL 형식의 문자열을 파이썬에서 처리할 수 있도록 리스트나 딕셔너리 형태로 변환하거나 그 반대 작업을 하기위해 import 한다.  
`import json`

In [4]:
final_docs = []

# JSONL 파일을 읽기 위해서 final_docs.jsonl 파일을 읽어들인다.
# './data' 폴더의 'final_docs.jsonl' 파일을 바이너리 형태의 입력용으로 open 한다.
# 파일 모드는 인코딩 문제를 방지하기 위해 'rb'는 Read Binary의 약자인 '읽기 전용' 및 '바이너리 모드'로 연다.
# with 구문을 사용했으므로 with 블록의 모든 작업이 완료되면 파일이 자동으로 안전하게 닫힌다. 입력용 파일을 닫는 이유는 메모리를 효율적으로 사용하기 위해서이다.
with open('./data/final_docs.jsonl', 'rb') as file:
    for line in file:
        # json 라이브러리의 loads() 메소드는 인수로 지정된 문자열을 리스트나 딕셔너리 형태로 변환한다.
        item = json.loads(line)
        # 딕셔너리에서 'page_content'와 'metadata'를 꺼내서 Document 객체를 만들어 final_docs 리스트에 추가한다.
        doc = Document(page_content=item['page_content'], metadata=item['metadata'])
        final_docs.append(doc)

print(len(final_docs))
final_docs

6


[Document(metadata={'source': './data\\리비안_KR.txt', 'doc_id': 0}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다\n2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다\n주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'),
 Document(metadata={'source': './data\\리비안_KR.txt', 'doc_id': 1}, page_content='리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다\n이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'),
 Document(metadata={'source': './data\\리비안_KR.txt', 'doc_id': 2}, page_content='리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다\n2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다\n리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'),
 Document(metadata={'source': './data\\테슬라_KR.txt', 'doc_id': 3}, page_content='테슬라(Tesla, Inc\n)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니

## 엑셀 파일 읽기

판다스를 사용해서 엑셀 파일을 불러온다.

`import pandas as pd`

In [5]:
# 판다스가 제공하는 read_excel() 메소드로 엑셀 파일을 읽어온다. openpyxl 라이브러리가 설치되어있어야 정상적으로 동작된다.
df_qa_test = pd.read_excel('./data/qa_test.xlsx')
df_qa_test.head(6)

,context,source,doc_id,question,answer
0,"['.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2...",['data/리비안_KR.txt'],['0'],리비안의 초기 모델은 무엇인가요?,리비안의 초기 모델은 스포츠카 R1입니다.
1,"['.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2...",['data/리비안_KR.txt'],['0'],R1의 좌석 구성은 어떻게 되나요?,R1은 2+2 좌석 구성입니다.
2,"['.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2...",['data/리비안_KR.txt'],['0'],R1은 어떤 구조를 특징으로 하나요?,R1은 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 합니다.
3,"['테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전...",['data/테슬라_KR.txt'],['1'],테슬라는 어디에 본사를 두고 있나요?,테슬라는 텍사스주 오스틴에 본사를 두고 있습니다.
4,"['테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전...",['data/테슬라_KR.txt'],['1'],테슬라는 언제 설립되었나요?,테슬라는 2003년에 설립되었습니다.
5,"['테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전...",['data/테슬라_KR.txt'],['1'],테슬라의 공동 창립자는 누구인가요?,테슬라의 공동 창립자는 마틴 에버하드와 마크 타페닝입니다.


# 검색기를 정의한다.

## 벡터검색기 - 시맨틱 검색, 맥락의 의미

임베딩 기술을 사용하여 텍스트의 의미를 숫자로 변환하고, 이를 바탕으로 질문과 가장 유사한 답변을 찾아내는 벡터검색기를 정의한다.

오픈소스 벡터 데이터이터베이스로 텍스트를 숫자로 바꾼 데이터를 저장하고 빠르게 검색하기 위해 Chroma를 import 한다.  
`from langchain_chroma import Chroma`  
오픈소스 모델 허브인 Hugging Face에서 제공하는 임베딩 모델을 사용하기 위해서 HuggingFaceEmbeddings를 import 한다.  
`from langchain_huggingface.embeddings import HuggingFaceEmbeddings`

벡터저장소에 사용한 임베딩 모델을 설정한다.

In [6]:
# HuggingFaceEmbeddings 클래스의 생성자로 임베딩 모델 이름을 넘겨서 임베딩 모델을 설정한다.
# 임베딩이란 컴퓨터 문장을 이해할 수 있는 숫자들로 바꾸는 과정이다.
embeddings_model = HuggingFaceEmbeddings(model='BAAI/bge-m3')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

벡터저장소를 로드한다.

In [7]:
# Chroma 클래스의 생성자로 임베딩 모델, 테이블 이름, 저장된 폴더를 넘겨서 벡터스토어를 로드한다.
chroma_db = Chroma(
    collection_name='hf_bge_m3',
    embedding_function=embeddings_model,
    persist_directory='./chroma_db',
)

벡터검색기를 만든다.

In [8]:
# as_retriever() 메소드로 질문과 유사도를 계산해서 의미적으로 가까운 가져올 문서 개수를 넘겨서 벡터검색기를 만든다.
chroma_k_retriever = chroma_db.as_retriever(search_kwargs={'k': 2})

벡터검색기를 실행한다.

In [9]:
query = '테슬라의 회장은 누구인가요?'
# invoke() 메소드로 쿼리를 넘겨서 거리가 가장 가까운 문서를 찾아 반환한다.
retriever_docs = chroma_k_retriever.invoke(query)

In [10]:
print(f'쿼리: {query}')
print('검색 결과')
for doc in retriever_docs:
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 100)

쿼리: 테슬라의 회장은 누구인가요?
검색 결과
테슬라(Tesla, Inc
)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다
2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'doc_id': 3, 'source': './data\\테슬라_KR.txt'}
----------------------------------------------------------------------------------------------------
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다
회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다
테슬라는 2010년 6월 나스닥에 상장되었습니다
2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37
65% 증가했습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'source': './data\\테슬라_KR.txt', 'doc_id': 4}
----------------------------------------------------------------------------------------------------


## BM25검색기 - 키워드 기반, 단어의 일치

한국어 형태소 분석기인 Kiwi를 활용하여, 텍스트의 키워드가 얼마나 일치하는지를 기준으로 문서를 찾아주는 KiWiBM25 검색기를 정의한다.

한국어 문장을 형태소 단위로 나눠주는 도구인 KiwiTokenizer를 import 한다.  
한국어 형태소 분석기인 Kiwi를 활용해서 관련 문서를 찾아주는 KiWiBM25RetrieverWithScore를 import 한다.

In [11]:
from krag.tokenizers import KiwiTokenizer
from krag.retrievers import KiWiBM25RetrieverWithScore

한국어를 분석에 사용할 토크나이저를 설정한다.

In [12]:
# KiwiTokenizer 클래스의 생성자로 토크나이저 모델과 오타 교정 여부를 넘겨서 토크나이저를 설정한다.
kiwi_tokenizer = KiwiTokenizer(
    # krag에서 가장 권장되는 모델인 특정 단어 뒤에 어떤 형태소가 나올 확률이 높은지 계산해서 문맥을 파악하는 knlm을 지정한다.
    model_type='knlm',
    # 형태소 분석시 일반적인 수준의 오타를 처리하는 'basic'을 지정한다.
    typos='basic'
)

KiWiBM25 검색기를 만든다.

In [13]:
# KiWiBM25RetrieverWithScore 클래스의 생성자로 토큰화할 문서, kiwi 토크나이저, 가져올 문서 개수, 검색 점수 임계값을 넘겨서 KiWiBM25 검색기를 만든다.
kiwibm25_k_retriever = KiWiBM25RetrieverWithScore(
    # KiWiBM25 검색기를 만들기 위해서 토큰화 할 문서 리스트를 지정한다.
    documents=final_docs,
    # 형태소 분석에 사용할 Kiwi 토크나이저를 지정한다.
    kiwi_tokenizer=kiwi_tokenizer,
    # 검색해서 가장 관련성이 높은 점수 순서로 가져올 문서의 개수를 지정한다.
    k=2,
    # 검색 점수의 임계값을 지정한다. 0.0으로 지정하면 점수에 상관없이 모든 결과를 가져온다.
    threshold=0.0
)

KiWiBM25 검색기를 실행한다.

In [14]:
query = '테슬라의 회장은 누구인가요?'
retriever_docs = kiwibm25_k_retriever.invoke(query)

In [15]:
print(f'쿼리: {query}')
print('검색 결과')
for doc in retriever_docs:
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 100)

쿼리: 테슬라의 회장은 누구인가요?
검색 결과
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다
회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다
테슬라는 2010년 6월 나스닥에 상장되었습니다
2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37
65% 증가했습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'source': './data\\테슬라_KR.txt', 'doc_id': 4, 'bm25_score': 2.1003500487956686}
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다
이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'source': './data\\리비안_KR.txt', 'doc_id': 1, 'bm25_score': 0.8556885950901274}
----------------------------------------------------------------------------------------------------


## Ensemble 검색기 - Hybrid Search

서로 다른 2개 이상의 검색기(Chroma(의미 기반), KiwiBM25(키워드 기반))를 하나로 합쳐서 각각의 장점만 취하는 Ensemble 검색기를 정의한다.

여러 개의 검색 알고리즘을 결합하여 더 나은 검색 결과를 만들어내는 EnsembleRetriever를 import 한다.

In [16]:
from langchain.retrievers import EnsembleRetriever

Ensemble 검색기를 만든다.

In [17]:
# Chroma 벡터저장소나 KiwiBM25 저장소를 새로 만들지 않고 기존 저장소의 검색기의 상위 k을 아래와 같이 변경할 수 있다.
chroma_k_retriever.search_kwargs['k'] = 4
kiwibm25_k_retriever.k = 4

In [18]:
# EnsembleRetriever 클래스의 생성자로 사용할 서로 다른 검색기 목록과 각 검색기의 비중을 넘겨서 Ensemble 검색기를 만든다.
ensemble_k_retriever = EnsembleRetriever(
    # 사용할 서로 다른 검색기 목록을 리스트 형태로 지정한다.
    retrievers=[chroma_k_retriever, kiwibm25_k_retriever],
    # 각 검색기별 가중치를 리스트 형태로 지정한다. 이 수치를 조절해서 검색 성능을 튜닝할 수 있다.
    weights=[0.5, 0.5]
)

Ensemble 검색기를 실행한다.

In [19]:
query = '테슬라의 회장은 누구인가요?'
retriever_docs = ensemble_k_retriever.invoke(query)

In [20]:
print(f'쿼리: {query}')
print('검색 결과')
for doc in retriever_docs:
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 100)

쿼리: 테슬라의 회장은 누구인가요?
검색 결과
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다
회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다
테슬라는 2010년 6월 나스닥에 상장되었습니다
2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37
65% 증가했습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'source': './data\\테슬라_KR.txt', 'doc_id': 4}
----------------------------------------------------------------------------------------------------
2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다
SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12
9%를 차지했습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'doc_id': 5, 'source': './data\\테슬라_KR.txt'}
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다
2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다
주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'source': './data\\리비안_KR.txt', 'doc_id': 0}
-------------------------------------------------------------------